# Chương 5 - Knowledge Graph và RAG

<span style='color:#c62828'><b>PHẦN CẬP NHẬT MỚI.</b> Notebook đi từ dữ liệu, biểu diễn hàm, code từng thành phần đến kết quả benchmark. Các cell chạy từ trên xuống.</span>

## 1. Khởi tạo

Notebook dùng đúng corpus, graph và benchmark trong repository; không lấy nhãn test để xếp hạng.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project:', ROOT)

Project: C:\DATA\assignment3\tieuluan3


## 2. Ba lớp dữ liệu

Hai CSV Diabetes và AAPL là bảng quan sát. `corpus.json` là kho văn bản có nguồn. `graph.json` lưu quan hệ và `benchmark.json` chỉ dùng để đánh giá sau retrieval.

In [2]:
corpus=json.loads((ROOT/'knowledge/corpus.json').read_text(encoding='utf-8'))
triples=json.loads((ROOT/'knowledge/graph.json').read_text(encoding='utf-8'))
queries=json.loads((ROOT/'knowledge/benchmark.json').read_text(encoding='utf-8'))
print('Documents:',len(corpus),'| Triples:',len(triples),'| Queries:',len(queries))
print('Domains:',{d:sum(x['domain']==d for x in corpus) for d in ['health','finance']})

Documents: 12 | Triples: 23 | Queries: 15
Domains: {'health': 6, 'finance': 6}


## 3. Knowledge Graph dưới dạng triple

Mỗi cạnh có `subject`, `relation`, `object` và `evidence`. Trường evidence nối quan hệ về đúng document có URL.

In [3]:
for t in triples[:10]: print(f"({t['subject']}) -[{t['relation']}]-> ({t['object']})  [{t['evidence']}]")

(đái tháo đường) -[có loại]-> (type 1)  [health_overview]
(đái tháo đường) -[có loại]-> (type 2)  [health_overview]
(đái tháo đường) -[có thể có dấu hiệu]-> (khát nhiều)  [health_symptoms]
(đái tháo đường) -[có thể có dấu hiệu]-> (tiểu nhiều)  [health_symptoms]
(đái tháo đường) -[có thể gây]-> (tổn thương thận)  [health_overview]
(đái tháo đường) -[được đánh giá bằng]-> (A1C)  [health_testing]
(đái tháo đường) -[được đánh giá bằng]-> (glucose huyết tương)  [health_testing]
(type 2) -[có thể phòng ngừa bằng]-> (vận động thể lực)  [health_prevention]
(type 2) -[có thể phòng ngừa bằng]-> (dinh dưỡng lành mạnh)  [health_prevention]
(CDC Diabetes) -[được dùng bởi]-> (MLP)  [health_dataset]


## 4. Hệ thống dưới dạng hợp các hàm

Với câu hỏi $q$:  \n$z=N(q)$ chuẩn hóa; $r_b=B(z)$ là BM25; $r_l=L(z)$ là LSA; $r_g=G(z,h)$ là graph retrieval tối đa $h$ bước; $r=F(r_b,r_l,r_g)$ dùng weighted RRF; cuối cùng $a=A(q,C(r))$ tạo đáp án từ context có nguồn.  \nTách hàm giúp chỉ rõ lỗi nằm ở chuẩn hóa, retrieval, graph, hợp nhất hay generation.

## 5. Khởi tạo engine

Constructor lập chỉ mục BM25, ma trận TF-IDF, phép chiếu SVD và adjacency list. Code đầy đủ nằm trong `src/knowledge_rag.py`.

In [4]:
from knowledge_rag import KnowledgeRAG
engine=KnowledgeRAG(ROOT/'knowledge')
print('TF-IDF:',engine.tfidf.shape,'| Dense:',engine.dense.shape)

TF-IDF: (12, 307) | Dense: (12, 8)


## 6. So sánh năm bộ truy xuất

BM25 dùng từ vựng; LSA dùng cosine trong không gian SVD; KG dùng entity và BFS; Hybrid hợp BM25+LSA; GraphRAG dùng đủ ba nhánh.

In [5]:
question='AAPL liên hệ với Apple, OHLCV và RNN như thế nào?'
for method in ['BM25','LSA','KG','Hybrid','GraphRAG']:
    r=engine.retrieve(question,method=method,top_k=3,hops=2)
    print(method,'->',[d['id'] for d in r['documents']])

BM25 -> ['stock_dataset', 'apple_ir', 'stock_baseline']
LSA -> ['stock_ohlcv', 'apple_ir', 'stock_return']
KG -> ['apple_ir', 'stock_baseline', 'stock_ohlcv']
Hybrid -> ['apple_ir', 'stock_dataset', 'stock_return']
GraphRAG -> ['apple_ir', 'stock_baseline', 'stock_dataset']


## 7. Đáp án có nguồn và đường graph

Đầu ra giữ URL, document id và các cạnh đã đóng góp. Generation offline chỉ ghép bằng chứng, tránh phụ thuộc API trả phí.

In [6]:
answer=engine.answer('AAPL có baseline nào và RNN có phải khuyến nghị đầu tư không?')
print(answer['answer'])
print('Sources:',[x['id'] for x in answer['citations']])
print('Paths:',answer['paths'][:4])

Mốc Last value dự báo giá phiên tiếp theo bằng giá Close gần nhất. Trên test AAPL của tiểu luận, mô hình chuỗi chưa tạo cải thiện có ý nghĩa thực hành so với mốc này. R-squared cao chủ yếu phản ánh chuỗi mức giá trơn và không chứng minh lợi thế đầu tư. Dự báo từ mô hình RNN là kết quả thực nghiệm trên dữ liệu lịch sử, không phải cam kết về giá tương lai hoặc khuyến nghị mua bán. Một câu hỏi về giá chính xác trong tương lai không có bằng chứng quan sát và Agent phải nêu thiếu dữ liệu thay vì tạo con số. Dataset AAPL có 2766 phiên từ 2015-01-02 đến 2025-12-31 với Date, Open, High, Low, Close, Adj Close và Volume. Bài tạo cửa sổ 30 phiên và tám đặc trưng từ OHLCV để dự báo log-return phiên kế tiếp rồi đổi về giá USD.
Sources: ['stock_baseline', 'stock_limits', 'stock_dataset']
Paths: [{'subject': 'AAPL', 'relation': 'có quan sát', 'object': 'OHLCV', 'evidence': 'stock_dataset', 'hop': 1}, {'subject': 'AAPL', 'relation': 'có baseline', 'object': 'Last value', 'evidence': 'stock_baseline', 

## 8. Kết quả benchmark

Mười hai câu có gold dùng Recall@3, MRR@3 và Complete@3. Ba câu ngoài phạm vi đo khả năng trả rỗng riêng; không trộn vào recall.

In [7]:
import pandas as pd
from IPython.display import display,Image
comparison=pd.read_csv(ROOT/'results/knowledge_rag/comparison.csv')
display(comparison.round(4))
display(Image(filename=ROOT/'results/knowledge_rag/retrieval_comparison.png'))

     Method  Recall@3   MRR@3  Complete@3  OOD_EmptyRate  Latency_ms
0      BM25    0.9167  0.9583      0.8333         0.0000      0.7185
1       LSA    0.8958  0.9583      0.7500         0.0000      0.7635
2        KG    0.7083  0.5972      0.6667         0.6667      0.8405
3    Hybrid    0.9375  0.9583      0.8333         0.0000      0.7876
4  GraphRAG    0.9167  0.9583      0.8333         0.0000      0.7912
<IPython.core.display.Image object>


## 9. Nhận xét

Corpus chỉ có 12 đoạn và benchmark do cùng dự án biên soạn. Điểm cao không chứng minh chuyển miền. Cơ chế từ chối còn yếu vì câu hỏi tương lai vẫn chia sẻ từ khóa với tài liệu lịch sử. Khi mở rộng cần corpus độc lập, người gán gold khác và đánh giá citation faithfulness.